In [ ]:
import json
import math
import time
import numpy as np
import pandas as pd
import os
from random import randint
from datetime import datetime, timedelta

import torch
import torch.optim as optim
import torch.utils.data as data
import matplotlib.pyplot as plt

import joblib

from utils.reproducibility import seed_everything, tf_func, clip_grad_func
from utils.loss_fn import masked_mae_multi as loss_fn
from utils.create_dataset_v1 import make_input_data, StationDataset
from utils.models import (AttrSeq2SeqLSTM as seq2seq_model, 
                          AttrSeq2SeqLSTM_v1 as seq2seq_model_v1,
                          AttrSeq2SeqCNNLSTM as seq2seq_cnn_model,
                          AttrLSTM as norm_model, 
                          AttrLSTMv1 as norm_model_v1, 
                          AttrSeq2SeqAttnLSTM as seq2seq_attn_model,
                          train_one_epoch, 
                          evaluate,
                          train_one_epoch_scaler,
                          evaluate_scaler,
                          model_run
                          )
from utils.utils import read_forecast_data

from sklearn.model_selection import train_test_split
from lstm1__const__ import DAYS, TRAIN_FILES, TEST_FILES

TEST_FILES = [
    ############ test stations #############
    '21027659'
] 
STATION_NAMES = {
    '21027659': 'P8'
}

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

INP_ATTRS = None
OUT_ATTRS = None

INP_ATTR_SIZE = None
OUT_ATTR_SIZE = None

USE_SCALER = True

print(f'Using device: {DEVICE}')
ENCODER_IN_DIM = 0
DECODER_TF_DIM = 0

SEED = 42
HIDDEN_DIM = 256

FORECAST_DATA = read_forecast_data('./data/forecast_hr.csv', device=DEVICE, use_scaler=USE_SCALER)
print('FORECAST_DATA.shape', FORECAST_DATA.shape)
# FORECAST_DATA: (T, n_days, day_dim) — forecast is grouped per day for per-step alignment
FORECAST_DAYS = FORECAST_DATA.shape[1]   # number of forecast days (e.g. 4)
FORECAST_DIM = FORECAST_DATA.shape[2]    # per-day feature dim (e.g. 10)
CNN_CONFIG = [[64, 128], 1, 1]

def _pick_best_epoch(val_losses, tol=0.0):
    v = np.asarray(val_losses, dtype=float)
    if v.size == 0:
        return None
    m = v.min()
    thresh = m * (1.0 + tol) if m > 0 else m + tol
    return int(np.argmax(v <= thresh))   # earliest epoch within tol (relative) of the min

def loadModel(DAY, loadbest=True):
    MODEL_LOADED = None
    epoch_files = {}
    for fname in os.listdir(f"models/{DAY}/"):
        if 'epoch_' not in fname:
            continue
        epoch_files[int(fname.split('epoch_')[1].split('.pth')[0])] = f"models/{DAY}/{fname}"
    filename = epoch_files[max(epoch_files)] if epoch_files else None
    def _load(path):
        try:
            return torch.load(path, map_location=DEVICE, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=DEVICE)
    if filename is not None:
        MODEL_LOADED = _load(filename)
        # Prefer the best-VALIDATION epoch (earliest at the min) over the last epoch, which for
        # these early-peaking models is the overfit tail. prev_losses is stored in the checkpoint.
        if loadbest:
            val = (MODEL_LOADED.get('prev_losses') or {}).get('test', {}).get('mean')
            if val:
                best = _pick_best_epoch(val)
                if best in epoch_files and epoch_files[best] != filename:
                    filename = epoch_files[best]
                    MODEL_LOADED = _load(filename)
                    print(f'  loadbest: epoch {best} (val {val[best]:.5f}) instead of last {max(epoch_files)}')
        extra = MODEL_LOADED['extra']
        INP_ATTRS = extra['inputs']
        OUT_ATTRS = extra['outputs']
        ENCODER_IN_DIM = extra['encoder_in_dim']
        DECODER_TF_DIM = extra['decoder_tf_dim']
        USE_SCALER = extra['use_scaler']
        LOOKBACK = extra['lookback']
        LOOKFORWARD = extra['lookforward']
        DROPOUT = extra['dropout']
        USE_WAVELET = extra['use_wavelet'] if 'use_wavelet' in extra else USE_WAVELET
        WAVELET_LVL = extra['wavelet_lvl'] if 'wavelet_lvl' in extra else WAVELET_LVL
        STN_MATCH_ATTRIBUTE = extra['stn_match_attr'] if 'stn_match_attr' in extra else STN_MATCH_ATTRIBUTE
        cnn_config = extra['cnn_config'] if 'cnn_config' in extra else CNN_CONFIG
        forecast_dim = FORECAST_DIM if ('nwp_' in filename or '_fc' in filename) else None
        HIDDEN_DIM = extra['hidden_dim'] if 'hidden_dim' in extra else HIDDEN_DIM
        OUT_ATTR_SIZE = len(OUT_ATTRS)
        model = None
        print('model type:', extra['model_type'])
        if 'model_type' in extra and extra['model_type'] == 'seq2seq':
            model = seq2seq_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM)
        elif 'model_type' in extra and extra['model_type'] == 'seq2seq_v1':
            print(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, HIDDEN_DIM, forecast_dim, FORECAST_DAYS)
            model = seq2seq_model_v1(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM, forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS)
        elif 'model_type' in extra and extra['model_type'] == 'seq2seq_attn':
            model = seq2seq_attn_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM, forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS)
        elif 'model_type' in extra and extra['model_type'] == 'seq2seq_cnn':
            model = seq2seq_cnn_model(
                ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE,
                hidden_dim=HIDDEN_DIM,
                forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS,
                # CNN experiments (toggle here):
                #   current 3-layer pooling : cnn_channels=(64,128,256), pool_size=2, dilation=1
                #   2-layer pooling         : cnn_channels=(64,128),     pool_size=2, dilation=1
                #   dilated, no pooling     : cnn_channels=(64,128),     pool_size=1, dilation=2
                cnn_channels=cnn_config[0], pool_size=cnn_config[1], dilation=cnn_config[2],
            )
        elif 'model_type' in extra and extra['model_type'] == 'plain_lstm':
            model = norm_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM, forecast_dim=forecast_dim, forecast_days=FORECAST_DAYS)
        elif 'model_type' in extra and extra['model_type'] == 'v1':
            model = norm_model_v1(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM)
        else:
            model = norm_model(ENCODER_IN_DIM, DECODER_TF_DIM, OUT_ATTR_SIZE, hidden_dim=HIDDEN_DIM)

        # model.load_state_dict(MODEL_LOADED, strict=True)
        state_dict = MODEL_LOADED["model"]
        if any(k.startswith("_orig_mod.") for k in state_dict):
            state_dict = {k[len("_orig_mod."):]: v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=True)
        # _unwrap_model(model).load_state_dict(MODEL_LOADED["model"], strict=True)
        prev_losses = MODEL_LOADED["prev_losses"]

        print('Model loaded:', filename)
    model.to(DEVICE) 
    return {'model': model, 'model_info': MODEL_LOADED}
LOADED = 1
del LOADED

In [ ]:
try:
    print(LOADED)
except Exception:
    MODELS  = {}
    RESULTS = {}
    for i0 in DAYS:
        if i0 not in MODELS:
            MODELS[i0] = {}
            RESULTS[i0] = {}
        for i1 in DAYS[i0]:
            print(i0, i1, DAYS[i0][i1])
            MODELS[i0][i1] = loadModel(DAYS[i0][i1])
            RESULTS[i0][i1] = {}
    LOADED = 'MODEL AND DATA LOADED'


In [ ]:
DATA_DIR = 'model_data_1/'
FILE_SUFFIX = '_hr_avg.csv'
# TRAIN_FILES = [
#     ############ train stations ############
#     '21004880', '21024003', '21026652', 
#     '21027656', '21027657', '21027658', 
#     '21027660', '21027661', 
#     '21027662', '21027663', '21027665', '21040276', '21040277', 
#     '21040278', '21040279', '21040280', '23001059', 
#     '23003762', '23003763', '23003764', '23003765', '23003766', 

#     'S100', 'S102', 'S104', 'S106', 
#     'S108', 'S109', 
#     'S111', 'S115', 'S116', 'S117', 
#     'S122', 'S24', 
#     'S43', 'S44', 'S60', 
# ]
# TEST_FILES = [
#     ############ test stations #############
#     '21027659', # Pinnacle@ Duxton_Everton Lamppost #8
#     '23001060', # Fuhua School mid-lvl big podium #4
#     '21026653', # Pinnacle@ Duxton_Ground Along Road #5
#     'S121', # Old Choa Chu Kang Road
#     'S50', # Clementi Road
#     'S107', # East Coast Parkway
#     'S06', 

#     # # obsoletes: 'S24B', 'S96', 'S97'
# ] 


# # LOOKBACK_DAYS = 60
# # INTERVAL = 60 * 60
# # LOOKBACK_TIME = 60 * 60 * 24 * LOOKBACK_DAYS
# # LOOKBACK = round(LOOKBACK_TIME / INTERVAL)
# LOOKBACK = None

# # LOOKFORWARD_TIME = 60 * 60 * 24 * 7
# # LOOKFORWARD = round(LOOKFORWARD_TIME / INTERVAL)
# LOOKFORWARD = None

BATCH_SIZE = 256
BATCH_SIZE_EFF = 256
BATCH_ACCU = int(BATCH_SIZE_EFF / BATCH_SIZE)
BATCH_COUNT = int(BATCH_SIZE_EFF * math.floor(5000 / BATCH_SIZE_EFF))


# INP_ATTRS = None
# OUT_ATTRS = None

# INP_ATTR_SIZE = None
# OUT_ATTR_SIZE = None

# USE_SCALER = False

# print(f'Using device: {DEVICE}')
# ENCODER_IN_DIM = 0
# DECODER_TF_DIM = 0

# HIDDEN_DIM = 256


SEED = 42
STN_MATCH_ATTRIBUTE = 'lat_long_h'

ATTR_MATCH = {
    'temperature pt100': 'Temp',
    'relative humidity': 'RH',
    'wind speed': 'WSpd',
    'sin_wind': 'Wdir',
    'cos_wind': 'Wdir',
    'sin_dir': 'Wdir',
    'sin_dir': 'Wdir',
    'solar radiation': 'Sol',
    'pm1.0': 'pm1',
    'pm2.5': 'pm2p5',
    'pm10': 'pm10'
}

SCALER = {}
INP_ATTRS = ['Temperature PT100', 'Relative Humidity', 'Wind speed', 'sin_dir', 'cos_dir', 'Solar radiation']

for attr in INP_ATTRS:
    p = f"./scalers/{attr}.gz"
    if os.path.exists(p):
        SCALER[attr.lower()] = joblib.load(p)
        print(f'scaler file - OK: ./scalers/{attr}.gz')
    else:
        SCALER[attr.lower()] = None
        print(f'scaler file - not exist: ./scalers/{attr}.gz')

stn_match_file = open(DATA_DIR + 'station_match_1.json')
station_match = json.loads(stn_match_file.read())
stn_match_file.close()


In [ ]:
COMMON_KEY_IDX = 0

In [ ]:
EVAL_ORDER = ['Temp', 'RH', 'WSpd', 'WDir']
ATTR_LABEL_MATCH = {
    'Temp': 'Temperature',
    'RH': 'Relative Humidity',
    'WSpd': 'Wind Speed',
    'WDir': 'Wind Direction',
}

def createDataset(MODEL_DATA, station_idx, dataset_type='train', INCLUDE_YEAR=False):
    extra = MODEL_DATA['extra']
    OUT_ATTRS = extra['outputs']
    LOOKBACK = extra['lookback']
    LOOKFORWARD = extra['lookforward']
    USE_WAVELET = extra['use_wavelet'] if 'use_wavelet' in extra else USE_WAVELET
    WAVELET_LVL = extra['wavelet_lvl'] if 'wavelet_lvl' in extra else WAVELET_LVL
    BATCHNORM = extra['batchnorm'] if 'batchnorm' in extra else BATCHNORM
    STN_MATCH_ATTRIBUTE = extra['stn_match_attr'] if 'stn_match_attr' in extra else STN_MATCH_ATTRIBUTE
    # Must match training: if the model was trained without the absolute-year feature, zero it here
    # too. Default True for older checkpoints (trained with year_norm included).
    # INCLUDE_YEAR = extra.get('include_year', True)

    DATA_DIR = 'model_data_preprocessed_nosol/' + ATTR_MATCH[OUT_ATTRS[0]] + '_60/'

    if dataset_type == 'train' or dataset_type == 'eval':
        file = TRAIN_FILES[station_idx]
        stn_data = np.load(f'{DATA_DIR}/{file}.npz')
        stn_info = np.array(station_match[file][STN_MATCH_ATTRIBUTE], dtype=np.float32)
        time_feats, target, target_mask, valid_idx = stn_data['time_feats'], stn_data['target'], stn_data['target_mask'], stn_data['valid_idx']
        series = stn_data['series']
        inp = stn_data['input']
        inp_mask = stn_data['input_mask']
        forecast_idx = stn_data['forecast_idx']

        # Date-based split on the forecast target period: TEST = windows whose 168h target
        # starts on/after SPLIT_DATE; TRAIN = windows whose target ends before SPLIT_DATE.
        # Windows straddling the boundary are dropped so no training target overlaps the test
        # period. Exact for a Jan-1 cutoff: time_feats[:, 4] is the absolute year (years since
        # 1970, base_year=0), so the year comparison IS the date comparison.
        SPLIT_DATE = np.datetime64('2025-01-01')
        SPLIT_YEAR = int(SPLIT_DATE.astype('datetime64[Y]').astype(int))  # years since 1970; exact for a Jan-1 cutoff
        year_of = time_feats[:, 4]
        tgt_start_year = year_of[valid_idx + LOOKBACK]
        tgt_end_year   = year_of[valid_idx + LOOKBACK + LOOKFORWARD - 1]
        is_test  = tgt_start_year >= SPLIT_YEAR
        is_train = tgt_end_year   <  SPLIT_YEAR
        train_idx, train_forecast = valid_idx[is_train], forecast_idx[is_train]
        test_idx,  test_forecast  = valid_idx[is_test],  forecast_idx[is_test]

        if not INCLUDE_YEAR:
            time_feats[:, 4] = 0.0   # series[:, :5] IS time_feats, so zero both
            series[:, 4] = 0.0
        if dataset_type == 'train':
            return StationDataset(
            series, time_feats, target, target_mask, train_idx, stn_info, 
            LOOKBACK, LOOKFORWARD, forecast=FORECAST_DATA, forecast_idx=train_forecast,
            use_wavelet=USE_WAVELET, wavelet_level=WAVELET_LVL, inp=inp, inp_mask=inp_mask)
        if dataset_type == 'eval':
            return StationDataset(
            series, time_feats, target, target_mask, test_idx, stn_info,
            LOOKBACK, LOOKFORWARD, forecast=FORECAST_DATA, forecast_idx=test_forecast,
            use_wavelet=USE_WAVELET, wavelet_level=WAVELET_LVL, inp=inp, inp_mask=inp_mask)
    if dataset_type == 'test':
        file = TEST_FILES[station_idx]
        stn_data = np.load(f'{DATA_DIR}/{file}.npz')
        stn_data = np.load(f'{DATA_DIR}/{file}.npz')
        stn_info = np.array(station_match[file][STN_MATCH_ATTRIBUTE], dtype=np.float32)
        time_feats, target, target_mask, valid_idx = stn_data['time_feats'], stn_data['target'], stn_data['target_mask'], stn_data['valid_idx']
        series = stn_data['series']
        inp = stn_data['input']
        inp_mask = stn_data['input_mask']
        forecast_idx = stn_data['forecast_idx']
        if not INCLUDE_YEAR:
            time_feats[:, 4] = 0.0   # series[:, :5] IS time_feats, so zero both
            series[:, 4] = 0.0
        return StationDataset(
            series, time_feats, target, target_mask, valid_idx, stn_info,
            LOOKBACK, LOOKFORWARD, forecast=FORECAST_DATA, forecast_idx=forecast_idx,
            use_wavelet=USE_WAVELET, wavelet_level=WAVELET_LVL, inp=inp, inp_mask=inp_mask)


def _sample_start_keys(dataset):
    """Map each sample -> a key identifying its decoder start time (first entry of dec_tf).

    dec_tf[0] for sample i is time_feats[valid_idx[i] + lookback], which uniquely
    identifies the window's start time. valid_idx / time_feats do not depend on the
    model (wavelet or not), so a given start-time key maps to the same sample index
    across all models of the same attribute.
    Returns {start_key: sample_idx}.
    """
    starts = np.asarray(dataset.valid_idx) + dataset.lookback
    tf0_all = dataset.time_feats[starts].detach().cpu().numpy().astype(float)   # (N, 5)
    tf0_all = np.round(tf0_all, 6)
    return {tuple(row): i for i, row in enumerate(tf0_all)}


def _fmt_start_time(tf0):
    ms, mc, ds_, dc, yr = [float(v) for v in tf0]
    hour = (math.atan2(ms, mc) / (2 * math.pi)) % 1.0 * 24
    doy    = int(round((math.atan2(ds_, dc) / (2 * math.pi)) % 1.0 * 365))
    year   = int(round(1970 + yr))
    hh, mm = int(round(hour)), 0
    print(doy, year, hh, mm)
    date = datetime(year, 1, 1) + timedelta(days=doy - 1) + timedelta(hours=hh)
    return f'{date.strftime("%Y-%m-%d %H:%M")}'
    return f'{year} doy~{doy:.0f} {hh:02d}:{mm:02d}'


def eval_one_sample(dataset_type='train'):
    global COMMON_KEY_IDX
    # color and dash style follow the MODEL (entity), not plot order,
    # so each model keeps its look across every attribute panel.
    # Well-separated hues: blue, red, green, purple, orange, cyan, magenta, brown.
    _palette = ['#1f4fe0', '#e41a1c', '#2ca02c', '#8e2dc5', '#ff8c00', "#3b4445", '#e6007e', '#8b5a2b']
    _styles = [ '--', '-.', ':', (0, (5, 1, 1, 1)), (0, (3, 1))]
    # _styles = ['solid']
    model_colors = {name: _palette[i % len(_palette)] for i, name in enumerate(DAYS)}
    model_styles = {name: _styles[i % len(_styles)] for i, name in enumerate(DAYS)}
    n_attrs = len(EVAL_ORDER)
    fig, axes = plt.subplots(n_attrs, 1, figsize=(14, 4 * n_attrs))
    if n_attrs == 1:
        axes = [axes]

    if dataset_type == 'test':
        station_idx = randint(0, len(TEST_FILES) - 1)
        station_number = TEST_FILES[station_idx]
    else:
        station_idx = randint(0, len(TRAIN_FILES) - 1)
        station_number = TRAIN_FILES[station_idx]

    # --- Index each attribute's samples by decoder start time, then pick a start time
    #     that is common to ALL attributes so every panel shows the same window. ---
    attr_startkeys = {}   # attr -> {start_key: sample_idx}
    common_keys = None
    for attr in EVAL_ORDER:
        first_i0 = next((i0 for i0 in DAYS if attr in DAYS[i0]), None)
        if first_i0 is None:
            continue
        ds = createDataset(MODELS[first_i0][attr]['model_info'], station_idx, dataset_type, INCLUDE_YEAR=True)
        keys = _sample_start_keys(ds)
        attr_startkeys[attr] = keys
        common_keys = set(keys) if common_keys is None else (common_keys & set(keys))
    common_key = None
    if common_keys:
        common_keys = sorted(list(common_keys), key=lambda x: (x[4], math.atan2(x[2], x[3]), math.atan2(x[0], x[1])))  # sort by year, day, hour, minute
        # print(common_keys)

        # common_key = list(common_keys)[randint(0, len(common_keys) - 1)]
        # print(f'len(common_keys): {len(common_keys)}')
        # COMMON_KEY_IDX = 29940
        # common_key = list(common_keys)[COMMON_KEY_IDX % len(common_keys)] 
        # print(f'COMMON_KEY_IDX: {COMMON_KEY_IDX}')
        # print(f'common_key: {common_key}')
        # COMMON_KEY_IDX += 1

        # common_key = list(common_keys)[31 % len(common_keys)] 
        common_key = list(common_keys)[29936 % len(common_keys)] 
        
        fig.suptitle(f'Station {STATION_NAMES[station_number]} - start {_fmt_start_time(common_key)}', y=1.0)
    else:
        print('No start time common to all attributes for this station; '
              'falling back to a per-attribute random sample.')

    for ax, attr in zip(axes, EVAL_ORDER):
        # Resolve the sample index for this attribute at the shared start time.
        if common_key is not None and attr in attr_startkeys:
            sample_idx = attr_startkeys[attr][common_key]
        else:
            sample_idx = None
        # Build dataset once per attr using any model that has it
        first_i0 = next((i0 for i0 in DAYS if attr in DAYS[i0]), None)
        if first_i0 is None:
            ax.set_title(f'{attr} (no model)')
            continue


        y_plotted = False
        y0 = None
        for i0 in DAYS:
            if attr not in DAYS[i0]:
                continue
            dataset = createDataset(MODELS[i0][attr]['model_info'], station_idx, dataset_type, INCLUDE_YEAR=True)
            if sample_idx is None:
                sample_idx = randint(0, len(dataset) - 1)
            print(sample_idx, end=' ')
            enc_seq, dec_tf, mask, y, stn, last_val, forecast = dataset[sample_idx]
            enc_seq[:, 4] = 0.0   # series[:, :5] IS time_feats, so zero both
            dec_tf[:, 4] = 0.0

            mask_np = mask.cpu().numpy().astype(bool)
            if mask_np.ndim > 1:
                mask_np = mask_np[:, 0]

            model = MODELS[i0][attr]['model']
            model_data = MODELS[i0][attr]['model_info']
            out_attrs = model_data['extra']['outputs']
            use_scaler = model_data['extra']['use_scaler']
            scaler_key = out_attrs[0].lower()
            scaler = SCALER.get(scaler_key) if use_scaler else None
            model.eval()
            # print('......', 
            #       enc_seq.unsqueeze(0).shape, dec_tf.unsqueeze(0).shape,
            #       last_val.unsqueeze(0).shape,
            #       forecast.unsqueeze(0).shape
            # )
            # print(last_val)
            with torch.no_grad():
                pred = model_run(
                    model,
                    enc_seq.unsqueeze(0), dec_tf.unsqueeze(0),
                    last_val=last_val.unsqueeze(0),
                    station_feats=stn.unsqueeze(0),
                    forecast=forecast.unsqueeze(0)
                )

            pred_np = pred.squeeze(0).detach().cpu().float().numpy()
            y_np    = y.cpu().float().numpy()
            if pred_np.ndim > 1: pred_np = pred_np[:, 0]
            if y_np.ndim > 1:    y_np    = y_np[:, 0]

            if scaler is not None:
                pred_np = scaler.inverse_transform(pred_np.reshape(-1, 1)).ravel()
                y_np    = scaler.inverse_transform(y_np.reshape(-1, 1)).ravel()
            if y0 is None: y0 = y_np.reshape(-1)

            # MAE / RMSE over the valid (unmasked) timesteps, in the plotted units
            if mask_np.any():
                err  = pred_np[mask_np] - y_np[mask_np]
                mae  = np.mean(np.abs(err))
                rmse = np.sqrt(np.mean(err ** 2))
            else:
                mae = rmse = np.nan

            pred_masked = np.where(mask_np, pred_np, np.nan)

            if not y_plotted:
                ax.plot(np.where(mask_np, y_np, np.nan),
                        color='black', linewidth=1.5, label='True', zorder=5)
                y_plotted = True

            ax.plot(pred_masked, linewidth=1.6, color=model_colors[i0], linestyle=model_styles[i0],
                    label=f'{i0} (RMSE {rmse:.3f})')

        ax.set_title(f'{ATTR_LABEL_MATCH[attr]} (Station {STATION_NAMES[station_number]})')
        ax.set_xticks(np.arange(0, 169, 24))
        ax.grid(True, alpha=0.3)
        ax.legend()


    plt.tight_layout()
    plt.show()

eval_one_sample('test')